# Praktikum 03 — Pengelolaan Data dan Privasi
**Dataset:** Penjualan UMKM 2024  
**Nama:** Aqila Diliansyah Putra  
**NIM:** 0110225025

Notebook ini menggunakan dataset transaksi UMKM yang diunggah. Jalankan sel dari atas ke bawah. Sebelum membagikan dataset, periksa apakah ada informasi yang bisa mengungkap identitas pelanggan atau pelaku usaha.

In [1]:
import pandas as pd
import numpy as np
import hashlib
from google.colab import files

uploaded = files.upload()
FILE = next(iter(uploaded))
df = pd.read_csv(FILE)

print("Nama file:", FILE)
print("Ukuran data:", df.shape)
display(df.head())

Saving transaksi_koperasi_2025 (1).csv to transaksi_koperasi_2025 (1).csv
Nama file: transaksi_koperasi_2025 (1).csv
Ukuran data: (1543, 16)


,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir
0,TRX251385,2025-10-29 11:19:00,AGT1150,Mi Instan Rebus,Makanan,porsi,6,10000,0,60000,QRIS,Rina,392000.0,2.445,3,NaN
1,TRX251333,2025-09-29 11:46:00,AGT1259,Teh Botol,Minuman,botol,3,6000,0,Rp 18.000,Saldo Koperasi,Dimas,152000.0,0.583,5,Pelanggan tetap
2,TRX251278,2025-10-25 13:18:00,AGT1145,Map Plastik,ATK,pcs,4,4000,0,16000,Saldo Koperasi,Bagus,127000.0,2.181,5,Pelanggan tetap
3,TRX251304,2025-09-09 13:45:00,AGT1240,Fotokopi A4,Alat Tulis,pcs,1,500,0,250,Tunai,Rina,1008000.0,NaN,5,NaN
4,TRX251364,2025-08-09 10:48:00,AGT1191,Map Plastik,ATK,pcs,5,4000,5,19000,QRIS,Rina,139000.0,2.376,4,Pelanggan tetap


In [18]:
import pandas as pd
import numpy as np
import hashlib
from google.colab import files

uploaded = files.upload()
FILE = next(iter(uploaded))
df = pd.read_csv(FILE)

print("Nama file:", FILE)
print("Ukuran data:", df.shape)
display(df.head())

Saving Data Stunting Indonesia (2).csv to Data Stunting Indonesia (2).csv
Nama file: Data Stunting Indonesia (2).csv
Ukuran data: (38, 1)


,,,,Provinsi;2020;2021;2022;2023;
ACEH;13,4;12,1;8,0;7,5;
SUMATERA UTARA;6,8;6,7;5,5;4,5;
SUMATERA BARAT;17,5;15,1;10,3;8,8;
RIAU;7,7;6,0;4,2;3,0;
JAMBI;7,6;3,0;4,1;3,7;


## A. Pemeriksaan awal dataset

In [2]:
# A1 — Struktur dan tipe data
print("Ukuran (baris, kolom):", df.shape)
print("\nTipe data:")
display(df.dtypes.rename("tipe_data").to_frame())
print("\nNama kolom:")
print(df.columns.tolist())

Ukuran (baris, kolom): (1543, 16)

Tipe data:


,tipe_data
id_transaksi,object
tanggal_transaksi,object
id_anggota,object
nama_produk,object
kategori,object
satuan,object
jumlah,int64
harga_satuan,int64
diskon_persen,int64
total_bayar,object



Nama kolom:
['id_transaksi', 'tanggal_transaksi', 'id_anggota', 'nama_produk', 'kategori', 'satuan', 'jumlah', 'harga_satuan', 'diskon_persen', 'total_bayar', 'metode_bayar', 'kasir', 'saldo_simpanan', 'berat_kg', 'rating_layanan', 'catatan_kasir']


In [3]:
# A2 — Ringkasan kolom numerik dan kategorikal
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id_transaksi,1536,1493,TRX251348,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tanggal_transaksi,1543,1495,23/10/2025 09:52,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
id_anggota,1543,318,AGT1041,16,NaN,NaN,NaN,NaN,NaN,NaN,NaN
nama_produk,1543,18,Roti Isi,144,NaN,NaN,NaN,NaN,NaN,NaN,NaN
kategori,1543,18,Makanan,469,NaN,NaN,NaN,NaN,NaN,NaN,NaN
satuan,1543,5,porsi,508,NaN,NaN,NaN,NaN,NaN,NaN,NaN
jumlah,1543.0,NaN,NaN,NaN,4.117952,2.57904,-8.0,2.0,4.0,6.0,8.0
harga_satuan,1543.0,NaN,NaN,NaN,7891.769281,6549.147006,500.0,4000.0,6000.0,10000.0,35000.0
diskon_persen,1543.0,NaN,NaN,NaN,1.007777,6.628844,-20.0,0.0,0.0,0.0,150.0
total_bayar,1543,220,12000,79,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### A3 — Pertanyaan analisis
Contoh pertanyaan yang bisa dijawab dari kolom dataset:
1. Apakah jumlah barang berkaitan dengan total pembayaran?
2. Produk atau kategori apa yang memiliki total penjualan tertinggi?
3. Apakah nilai transaksi berbeda menurut metode pembayaran?

Catatan: data ini berisi catatan transaksi; tidak bisa dipakai untuk menyimpulkan perubahan omzet suatu usaha dari waktu ke waktu tanpa informasi tambahan yang sesuai.

In [4]:
# A4 — Temukan kolom yang berkaitan dengan transaksi atau pembayaran
[c for c in df.columns if any(k in c.lower() for k in ["total", "harga", "jumlah", "bayar"])]

['jumlah', 'harga_satuan', 'total_bayar', 'metode_bayar']

## B. Skala pengukuran dan tipe tanggal

In [5]:
# B1 — Tentukan skala pengukuran berdasarkan makna kolom
skala = {
    "id_transaksi": "Nominal (pengenal transaksi)",
    "tanggal": "Interval (jarak waktu bermakna)",
    "kota": "Nominal",
    "kategori": "Nominal",
    "produk": "Nominal",
    "jumlah": "Rasio",
    "harga_satuan": "Rasio",
    "diskon_persen": "Rasio",
    "metode_bayar": "Nominal",
    "rating_pelanggan": "Ordinal (rating berurutan)",
    "total_bayar": "Rasio"
}
tabel_skala = pd.DataFrame({
    "nama_kolom": df.columns,
    "skala_pengukuran": [skala.get(c, "Perlu ditentukan sesuai makna kolom") for c in df.columns]
})
display(tabel_skala)

,nama_kolom,skala_pengukuran
0,id_transaksi,Nominal (pengenal transaksi)
1,tanggal_transaksi,Perlu ditentukan sesuai makna kolom
2,id_anggota,Perlu ditentukan sesuai makna kolom
3,nama_produk,Perlu ditentukan sesuai makna kolom
4,kategori,Nominal
5,satuan,Perlu ditentukan sesuai makna kolom
6,jumlah,Rasio
7,harga_satuan,Rasio
8,diskon_persen,Rasio
9,total_bayar,Rasio


In [6]:
# B2 — Konversi tanggal dan pemeriksaan rentang
if "tanggal" in df.columns:
    df["tanggal"] = pd.to_datetime(df["tanggal"], errors="coerce")
    print("Tanggal paling awal:", df["tanggal"].min())
    print("Tanggal paling akhir:", df["tanggal"].max())
    print("Tanggal gagal dikonversi:", df["tanggal"].isna().sum())

## C. Kamus data dan pemeriksaan nilai kosong

In [7]:
# C1 — Bangkitkan kerangka kamus data dari dataset
kamus = pd.DataFrame({
    "nama_kolom": df.columns,
    "tipe_data": [str(df[c].dtype) for c in df.columns],
    "jumlah_nilai_unik": [df[c].nunique(dropna=True) for c in df.columns],
    "jumlah_kosong": [df[c].isna().sum() for c in df.columns],
    "contoh_nilai": [df[c].dropna().iloc[0] if df[c].notna().any() else np.nan for c in df.columns]
})
display(kamus)

,nama_kolom,tipe_data,jumlah_nilai_unik,jumlah_kosong,contoh_nilai
0,id_transaksi,object,1493,7,TRX251385
1,tanggal_transaksi,object,1495,0,2025-10-29 11:19:00
2,id_anggota,object,318,0,AGT1150
3,nama_produk,object,18,0,Mi Instan Rebus
4,kategori,object,18,0,Makanan
5,satuan,object,5,0,porsi
6,jumlah,int64,16,0,6
7,harga_satuan,int64,13,0,10000
8,diskon_persen,int64,6,0,0
9,total_bayar,object,220,0,60000


In [8]:
# C2 — Daftar kolom yang mengandung nilai kosong
kamus[kamus["jumlah_kosong"] > 0]

,nama_kolom,tipe_data,jumlah_nilai_unik,jumlah_kosong,contoh_nilai
0,id_transaksi,object,1493,7,TRX251385
10,metode_bayar,object,11,9,QRIS
12,saldo_simpanan,float64,624,159,392000.0
13,berat_kg,float64,978,351,2.445
15,catatan_kasir,object,4,1044,Pelanggan tetap


### C3 — Catatan kamus data
- `id_transaksi`: kode unik untuk membedakan transaksi.
- `tanggal`: tanggal transaksi dilakukan.
- `kota`: kota transaksi/pelanggan tercatat.
- `kategori` dan `produk`: pengelompokan barang yang dijual.
- `jumlah`: banyaknya unit dalam transaksi.
- `harga_satuan`: harga satu unit sebelum perhitungan total.
- `diskon_persen`: persentase diskon transaksi.
- `metode_bayar`: cara pembayaran.
- `rating_pelanggan`: penilaian pelanggan.
- `total_bayar`: jumlah pembayaran akhir yang tercatat.

Deskripsi ini berdasarkan nama kolom dan contoh data; konfirmasikan dengan pemilik dataset jika definisi bisnisnya berbeda.

## D. Identifikasi data pribadi dan strategi penanganan

In [9]:
# D1 — Penandaan kolom secara awal
# Ini merupakan evaluasi awal, bukan kepastian hukum atas klasifikasi data.
PENANDAAN = {
    "id_transaksi": ("pengenal transaksi", "hash"),
    "tanggal": ("quasi-identifier", "generalisasi"),
    "kota": ("quasi-identifier", "generalisasi"),
    "kategori": ("data transaksi", "pertahankan"),
    "produk": ("data transaksi", "pertahankan"),
    "jumlah": ("data transaksi", "pertahankan"),
    "harga_satuan": ("data transaksi", "pertahankan"),
    "diskon_persen": ("data transaksi", "pertahankan"),
    "metode_bayar": ("data transaksi", "pertahankan"),
    "rating_pelanggan": ("data transaksi", "pertahankan"),
    "total_bayar": ("data transaksi", "pertahankan")
}
tanda = pd.DataFrame([
    {"nama_kolom": c,
     "kategori": PENANDAAN.get(c, ("perlu ditinjau", "perlu ditinjau"))[0],
     "tindakan": PENANDAAN.get(c, ("perlu ditinjau", "perlu ditinjau"))[1]}
    for c in df.columns
])
display(tanda)

,nama_kolom,kategori,tindakan
0,id_transaksi,pengenal transaksi,hash
1,tanggal_transaksi,perlu ditinjau,perlu ditinjau
2,id_anggota,perlu ditinjau,perlu ditinjau
3,nama_produk,perlu ditinjau,perlu ditinjau
4,kategori,data transaksi,pertahankan
5,satuan,perlu ditinjau,perlu ditinjau
6,jumlah,data transaksi,pertahankan
7,harga_satuan,data transaksi,pertahankan
8,diskon_persen,data transaksi,pertahankan
9,total_bayar,data transaksi,pertahankan


### D2 — Alasan penandaan
`id_transaksi` diperlakukan sebagai pengenal unik dan dapat diganti dengan token hash. `tanggal` dan `kota` dapat menjadi quasi-identifier jika digabungkan dengan informasi lain. Kolom transaksi lain dipertahankan untuk analisis, tetapi perlu diperiksa kembali berdasarkan konteks, izin, dan risiko publikasi.

## E. Membuat salinan data yang lebih aman

In [10]:
# E1 — Terapkan penghapusan kolom yang memang ditandai 'hapus'
kolom_hapus = [c for c in df.columns if PENANDAAN.get(c, ("", "pertahankan"))[1] == "hapus"]
aman = df.drop(columns=kolom_hapus).copy()
print("Kolom yang dihapus:", kolom_hapus)
print("Ukuran data aman sementara:", aman.shape)

Kolom yang dihapus: []
Ukuran data aman sementara: (1543, 16)


In [11]:
# E2 — Hash pengenal transaksi
def samarkan(nilai, garam):
    if pd.isna(nilai):
        return np.nan
    return hashlib.sha256((garam + str(nilai)).encode("utf-8")).hexdigest()[:12]

# Ganti nilai ini sebelum dipakai; jangan simpan garam privat ke repositori publik.
GARAM = "ubah-menjadi-rahasia-lokal"
kolom_hash = [c for c in df.columns if PENANDAAN.get(c, ("", "pertahankan"))[1] == "hash"]

for c in kolom_hash:
    aman[c] = df[c].apply(lambda x: samarkan(x, GARAM))

print("Kolom yang di-hash:", kolom_hash)
if kolom_hash:
    display(aman[kolom_hash].head())

Kolom yang di-hash: ['id_transaksi']


,id_transaksi
0,9ed4505a6962
1,f1f5adee92a1
2,c2afb7552522
3,2abdeef689a1
4,38331cecd265


In [12]:
# E3 — Uji konsistensi hasil hash
if kolom_hash and len(df) >= 2:
    c = kolom_hash[0]
    h1 = samarkan(df[c].iloc[0], GARAM)
    h1_ulang = samarkan(df[c].iloc[0], GARAM)
    h2 = samarkan(df[c].iloc[1], GARAM)
    print("Nilai sama menghasilkan hash sama:", h1 == h1_ulang)
    print("Dua nilai berbeda menghasilkan hash berbeda:", h1 != h2)
else:
    print("Perlu kolom hash dan minimal dua baris untuk uji.")

Nilai sama menghasilkan hash sama: True
Dua nilai berbeda menghasilkan hash berbeda: True


### E4 — Generalisasi tanggal dan kota
Tanggal dapat diubah menjadi bulan atau tahun, sedangkan kota dapat digabung ke wilayah yang lebih luas bila informasi wilayah tersedia. Contoh berikut menghasilkan kolom bulan dan tidak mempertahankan tanggal persis.

In [13]:
if "tanggal" in aman.columns:
    aman["bulan_transaksi"] = pd.to_datetime(aman["tanggal"], errors="coerce").dt.to_period("M").astype(str)
    aman = aman.drop(columns=["tanggal"])
if "kota" in aman.columns:
    aman["kota"] = aman["kota"].astype("string").str.strip().str.title()

display(aman.head())

,id_transaksi,tanggal_transaksi,id_anggota,nama_produk,kategori,satuan,jumlah,harga_satuan,diskon_persen,total_bayar,metode_bayar,kasir,saldo_simpanan,berat_kg,rating_layanan,catatan_kasir
0,9ed4505a6962,2025-10-29 11:19:00,AGT1150,Mi Instan Rebus,Makanan,porsi,6,10000,0,60000,QRIS,Rina,392000.0,2.445,3,NaN
1,f1f5adee92a1,2025-09-29 11:46:00,AGT1259,Teh Botol,Minuman,botol,3,6000,0,Rp 18.000,Saldo Koperasi,Dimas,152000.0,0.583,5,Pelanggan tetap
2,c2afb7552522,2025-10-25 13:18:00,AGT1145,Map Plastik,ATK,pcs,4,4000,0,16000,Saldo Koperasi,Bagus,127000.0,2.181,5,Pelanggan tetap
3,2abdeef689a1,2025-09-09 13:45:00,AGT1240,Fotokopi A4,Alat Tulis,pcs,1,500,0,250,Tunai,Rina,1008000.0,NaN,5,NaN
4,38331cecd265,2025-08-09 10:48:00,AGT1191,Map Plastik,ATK,pcs,5,4000,5,19000,QRIS,Rina,139000.0,2.376,4,Pelanggan tetap


## F. Pemeriksaan risiko kelompok kecil (k-anonymity sederhana)

In [14]:
# F1 — Fungsi menghitung ukuran kelompok terkecil pada kombinasi quasi-identifier
def cek_k(data, kolom):
    tersedia = [c for c in kolom if c in data.columns]
    if not tersedia:
        return pd.Series({"k": np.nan, "jumlah_grup": 0, "grup_satu_record": 0})
    ukuran_grup = data.groupby(tersedia, dropna=False, observed=True).size()
    return pd.Series({
        "k": int(ukuran_grup.min()) if len(ukuran_grup) else np.nan,
        "jumlah_grup": len(ukuran_grup),
        "grup_satu_record": int((ukuran_grup == 1).sum())
    })

In [15]:
# F2 — Periksa kombinasi wilayah dan kategori yang tersedia
kolom_qi = [c for c in ["kota", "bulan_transaksi", "metode_bayar"] if c in aman.columns]
print("Kolom yang diperiksa:", kolom_qi)
display(cek_k(aman, kolom_qi))

Kolom yang diperiksa: ['metode_bayar']


,0
k,5
jumlah_grup,12
grup_satu_record,0


In [16]:
# F3 — Tabel agregat dan identifikasi sel berjumlah kecil
if "kota" in aman.columns and "kategori" in aman.columns:
    tabel_silang = pd.crosstab(aman["kota"], aman["kategori"])
    display(tabel_silang)
    sel_kecil = ((tabel_silang > 0) & (tabel_silang < 5)).sum().sum()
    print("Jumlah sel terisi dengan jumlah kurang dari 5:", int(sel_kecil))
else:
    print("Kolom kota/kategori tidak ditemukan.")

Kolom kota/kategori tidak ditemukan.


### F4 — Interpretasi
Nilai `k` yang kecil menunjukkan kombinasi atribut yang lebih jarang muncul sehingga berpotensi memudahkan pencocokan dengan data lain. Pemeriksaan ini hanya ukuran risiko sederhana; bukan jaminan anonimitas. Untuk publikasi, pertimbangkan menyamarkan atau menggabungkan kelompok kecil.

## G. Analisis ringkas dataset proyek

In [19]:
# G1 — Ringkasan nilai transaksi menurut kategori
if {"kategori", "total_bayar"}.issubset(df.columns):
    ringkasan_kategori = df.groupby("kategori", dropna=False)["total_bayar"].agg(
        jumlah_transaksi="count", total_penjualan="sum", rata_rata_transaksi="mean"
    ).sort_values("total_penjualan", ascending=False)
    display(ringkasan_kategori)

In [20]:
# G2 — Ringkasan menurut metode pembayaran
if {"metode_bayar", "total_bayar"}.issubset(df.columns):
    display(df.groupby("metode_bayar", dropna=False)["total_bayar"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False))

In [21]:
# G3 — Periksa hubungan numerik antarvariabel
kolom_numerik = df.select_dtypes(include="number").columns
display(df[kolom_numerik].corr(numeric_only=True))

""


### G4 — Pemeriksaan kualitas data
Periksa duplikasi, nilai kosong, dan rentang angka sebelum menarik kesimpulan. Hasil ini bersifat deskriptif dan tidak membuktikan hubungan sebab-akibat.

In [22]:
print("Ukuran data:", df.shape)
print("\nJumlah nilai kosong per kolom:")
display(df.isna().sum().rename("nilai_kosong").to_frame())
print("\nJumlah baris duplikat:", int(df.duplicated().sum()))
for c in df.select_dtypes(include="number").columns:
    print(f"{c}: min={df[c].min()}, max={df[c].max()}")

Ukuran data: (38, 1)

Jumlah nilai kosong per kolom:


,nilai_kosong
Provinsi;2020;2021;2022;2023;,0



Jumlah baris duplikat: 29


## G5 — Checklist sebelum Git Push
- [ ] Nama file dan sumber dataset sudah dicatat.
- [ ] Hak penggunaan/lisensi dataset sudah diperiksa.
- [ ] Pengenal unik sudah ditangani sesuai kebutuhan.
- [ ] Informasi sensitif dan quasi-identifier sudah ditinjau.
- [ ] Nilai kosong dan data duplikat sudah diperiksa.
- [ ] Definisi kolom sudah dicatat dalam kamus data.
- [ ] Tidak ada kata sandi, token, atau garam hash rahasia di repositori.
- [ ] Hasil analisis tidak ditafsirkan sebagai bukti sebab-akibat.